# Style-Matching Level Generation via KDE-Sampled GA

Generate novel Mummy Maze levels that match the style of the original 10,100 levels.

**Approach**: Fit a KDE to the original levels' feature vectors (BFS moves, n_states,
log win prob, wall count). For each new level to generate:
1. Sample a target feature vector from the KDE
2. Run a short GA (from random seeds) to produce a level matching that target
3. Keep the best result

This avoids the NN-convergence problem (where the GA just recreates originals) because
each target is a *noisy sample* from the distribution, not an exact original.

In [ ]:
import random
import numpy as np
import mummymaze_rust
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from scipy.stats import gaussian_kde
from IPython.display import HTML

MAZE_DIR = Path("../mazes")
GRID_SIZES = [6, 8, 10]

## 1. Compute feature vectors for all original levels

In [ ]:
def wall_count(level):
    """Count interior walls (excluding border) from edge arrays."""
    d = level.to_dict()
    gs = d["grid_size"]
    h = d["h_walls"]  # (gs+1) * gs bools — rows of horizontal edges
    v = d["v_walls"]  # gs * (gs+1) bools — rows of vertical edges
    # Interior h_walls: rows 1..gs-1 (exclude top border row 0 and bottom border row gs)
    h_interior = sum(h[i] for i in range(gs, gs * gs))  # skip first row of gs elements
    # Interior v_walls: for each row, columns 1..gs-1 (exclude left col 0 and right col gs)
    v_interior = 0
    for r in range(gs):
        for c in range(1, gs):
            v_interior += v[r * (gs + 1) + c]
    return h_interior + v_interior


def level_features(eval_result, level):
    """Extract feature vector from a ga_evaluate_batch result dict + level."""
    return np.array([
        eval_result["bfs_moves"],
        eval_result["n_states"],
        eval_result["log_win_prob"],
        wall_count(level),
    ], dtype=np.float64)


# Parse and evaluate all original levels
print("Parsing original levels...")
all_levels = {}
for gs in GRID_SIZES:
    all_levels[gs] = []

dat_files = sorted(MAZE_DIR.glob("B-*.dat"))
for dat_file in dat_files:
    try:
        levels = mummymaze_rust.parse_file(str(dat_file))
    except Exception:
        continue
    for lev in levels:
        d = lev.to_dict()
        gs = d["grid_size"]
        if gs in all_levels:
            all_levels[gs].append(lev)

for gs in GRID_SIZES:
    print(f"  gs={gs}: {len(all_levels[gs])} levels")

# Evaluate all original levels to get metrics
print("\nEvaluating original levels...")
original_features = {}
for gs in GRID_SIZES:
    results = mummymaze_rust.ga_evaluate_batch(all_levels[gs])
    feats = []
    for r in results:
        lev = r["level"]
        feats.append(level_features(r, lev))
    original_features[gs] = np.array(feats)
    print(f"  gs={gs}: {len(feats)} solvable levels, feature shape: {original_features[gs].shape}")

# Compute per-feature std for normalization
feature_scales = {}
for gs in GRID_SIZES:
    std = original_features[gs].std(axis=0)
    std[std < 1e-8] = 1.0  # avoid division by zero
    feature_scales[gs] = std
    print(f"  gs={gs} feature std: {std}")

## 2. Fit KDE + visualize the original distribution

In [ ]:
def random_level(grid_size, rng=random):
    """Generate a completely random level via from_edges()."""
    n = grid_size
    flip = rng.random() < 0.5

    # Random interior walls (~30% density)
    wall_prob = rng.uniform(0.15, 0.45)
    h_walls = []
    for r in range(n + 1):
        for c in range(n):
            if r == 0 or r == n:
                h_walls.append(True)
            else:
                h_walls.append(rng.random() < wall_prob)
    v_walls = []
    for r in range(n):
        for c in range(n + 1):
            if c == 0 or c == n:
                v_walls.append(True)
            else:
                v_walls.append(rng.random() < wall_prob)

    exit_side = rng.choice(["N", "S", "E", "W"])
    exit_pos = rng.randint(0, n - 1)

    occupied = set()
    def rand_pos():
        for _ in range(100):
            r, c = rng.randint(0, n - 1), rng.randint(0, n - 1)
            if (r, c) not in occupied:
                occupied.add((r, c))
                return (r, c)
        return (0, 0)

    player = rand_pos()
    mummy1 = rand_pos()
    mummy2 = rand_pos() if rng.random() < 0.4 else None
    scorpion = rand_pos() if rng.random() < 0.3 else None
    traps = []
    if rng.random() < 0.2:
        traps.append(rand_pos())
    if rng.random() < 0.1:
        traps.append(rand_pos())
    gate = None
    key = None
    if rng.random() < 0.15:
        gate = rand_pos()
        key = rand_pos()

    return mummymaze_rust.Level.from_edges(
        grid_size, flip, h_walls, v_walls,
        exit_side, exit_pos, player, mummy1,
        mummy2, scorpion, traps, gate, key,
    )


# Fit KDE per grid size
kdes = {}
for gs in GRID_SIZES:
    feats = original_features[gs]
    # gaussian_kde expects shape (n_features, n_samples)
    kdes[gs] = gaussian_kde(feats.T)
    print(f"gs={gs}: KDE fitted on {feats.shape[0]} levels, "
          f"bandwidth factor: {kdes[gs].factor:.4f}")

# Quick test: sample from the KDE and check they look reasonable
print("\nSample targets from KDE (gs=8):")
samples = kdes[8].resample(5).T
feature_names = ["BFS moves", "N states", "Log win prob", "Wall count"]
for i, s in enumerate(samples):
    print(f"  Sample {i}: " + ", ".join(f"{n}={v:.1f}" for n, v in zip(feature_names, s)))

In [ ]:
# Visualize 2D slices of the KDE for gs=8
gs = 8
feats = original_features[gs]
feature_names = ["BFS moves", "N states", "Log win prob", "Wall count"]
pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (i, j) in enumerate(pairs):
    ax = axes[idx]

    # Fit a 2D KDE on just this pair of features
    data_2d = feats[:, [i, j]].T
    kde_2d = gaussian_kde(data_2d)

    # Evaluate on a grid
    xi = np.linspace(feats[:, i].min() - feats[:, i].std(),
                     feats[:, i].max() + feats[:, i].std(), 80)
    yi = np.linspace(feats[:, j].min() - feats[:, j].std(),
                     feats[:, j].max() + feats[:, j].std(), 80)
    Xi, Yi = np.meshgrid(xi, yi)
    positions = np.vstack([Xi.ravel(), Yi.ravel()])
    Zi = kde_2d(positions).reshape(Xi.shape)

    # Plot density as filled contours
    ax.contourf(Xi, Yi, Zi, levels=20, cmap="YlOrRd")
    ax.scatter(feats[:, i], feats[:, j], s=1, alpha=0.15, color="black")
    ax.set_xlabel(feature_names[i])
    ax.set_ylabel(feature_names[j])
    ax.set_title(f"{feature_names[i]} vs {feature_names[j]}")

fig.suptitle("2D KDE Slices of Original Level Distribution (gs=8)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. KDE-sampled GA: sample a target, evolve toward it

In [ ]:
def target_fitness(feat, target, scales):
    """Negative normalized distance to a specific target vector."""
    return -np.sqrt(((feat - target) / scales) ** 2).sum()


def generate_one_level(grid_size, target, kde, scales, pop_size=64, generations=30, rng=random):
    """Run a short GA to produce a level matching a target feature vector."""
    # Generate initial population
    population = []
    for _ in range(20):  # try up to 20 rounds to fill population
        batch = [random_level(grid_size, rng) for _ in range(pop_size * 3)]
        results = mummymaze_rust.ga_evaluate_batch(batch)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            fit = target_fitness(feat, target, scales)
            population.append((lev, feat, fit))
            if len(population) >= pop_size:
                break
        if len(population) >= pop_size:
            break

    if not population:
        return None

    population.sort(key=lambda x: x[2], reverse=True)

    for gen in range(generations):
        n_elite = max(1, int(len(population) * 0.15))
        next_gen = list(population[:n_elite])

        offspring_levels = []
        while len(offspring_levels) < pop_size - n_elite:
            candidates = rng.sample(population, min(3, len(population)))
            parent = max(candidates, key=lambda x: x[2])
            if rng.random() < 0.8:
                child = mummymaze_rust.mutate(parent[0], rng.randint(0, 2**63))
            else:
                candidates2 = rng.sample(population, min(3, len(population)))
                parent2 = max(candidates2, key=lambda x: x[2])
                mode = rng.choice(["swap_entities", "region", "wall_patch", "feature_level"])
                child = mummymaze_rust.ga_crossover(parent[0], parent2[0], mode, rng.randint(0, 2**63))
            offspring_levels.append(child)

        results = mummymaze_rust.ga_evaluate_batch(offspring_levels)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            fit = target_fitness(feat, target, scales)
            next_gen.append((lev, feat, fit))

        next_gen.sort(key=lambda x: x[2], reverse=True)
        population = next_gen[:pop_size]

    return population[0]  # (level, features, fitness)


def generate_levels_kde(grid_size, n_levels=50, pop_size=64, generations=30, seed=42):
    """Generate n_levels by repeatedly sampling KDE targets and running short GAs."""
    rng = random.Random(seed)
    np_rng = np.random.RandomState(seed)
    kde = kdes[grid_size]
    scales = feature_scales[grid_size]

    generated = []
    for i in range(n_levels):
        # Sample a target from the KDE
        target = kde.resample(1, seed=np_rng).flatten()

        # Run a short GA toward this target
        result = generate_one_level(grid_size, target, kde, scales,
                                     pop_size=pop_size, generations=generations, rng=rng)
        if result is not None:
            lev, feat, fit = result
            generated.append((lev, feat, fit, target))
            if (i + 1) % 10 == 0:
                print(f"  Generated {i+1}/{n_levels}, latest fitness: {fit:.3f}")

    print(f"  Done: {len(generated)}/{n_levels} levels generated")
    return generated

In [ ]:
# Generate 50 levels for gs=8
print("Generating levels for gs=8...")
generated_8 = generate_levels_kde(8, n_levels=50, pop_size=64, generations=30, seed=42)

## 4. Maze renderer + BFS solution animation

In [ ]:
ENTITY_RADIUS = 0.3
TRAP_SIZE = 0.25

def draw_maze(ax, level_dict, frame=None, title=None):
    """Draw a maze state on a matplotlib axes.

    Args:
        ax: matplotlib axes
        level_dict: from level.to_dict()
        frame: optional state dict from replay_actions (entity positions for this step)
        title: optional title string
    """
    ax.clear()
    gs = level_dict["grid_size"]

    # Grid coordinates: column = x, row = y (inverted so row 0 is at top)
    ax.set_xlim(-0.5, gs - 0.5)
    ax.set_ylim(gs - 0.5, -0.5)  # inverted y
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    # Background
    ax.set_facecolor("#f5f0e0")

    # Draw grid cells
    for r in range(gs):
        for c in range(gs):
            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                  linewidth=0.5, edgecolor="#d0c8b0",
                                  facecolor="#f5f0e0")
            ax.add_patch(rect)

    # Draw walls from h_walls / v_walls
    h_walls = level_dict["h_walls"]  # (gs+1) rows, gs cols
    v_walls = level_dict["v_walls"]  # gs rows, (gs+1) cols
    wall_color = "#2a1a0a"
    wall_lw = 3.0

    for r in range(gs + 1):
        for c in range(gs):
            if h_walls[r * gs + c]:
                # Horizontal wall on top edge of row r (between rows r-1 and r)
                ax.plot([c - 0.5, c + 0.5], [r - 0.5, r - 0.5],
                        color=wall_color, linewidth=wall_lw, solid_capstyle="round")

    for r in range(gs):
        for c in range(gs + 1):
            if v_walls[r * (gs + 1) + c]:
                # Vertical wall on left edge of col c (between cols c-1 and c)
                ax.plot([c - 0.5, c - 0.5], [r - 0.5, r + 0.5],
                        color=wall_color, linewidth=wall_lw, solid_capstyle="round")

    # Draw exit opening (erase the border wall segment)
    exit_side = level_dict["exit_side"]
    exit_pos = level_dict["exit_pos"]
    exit_color = "#4CAF50"
    exit_lw = 4.0
    if exit_side == "N":
        ax.plot([exit_pos - 0.4, exit_pos + 0.4], [-0.5, -0.5],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "S":
        ax.plot([exit_pos - 0.4, exit_pos + 0.4], [gs - 0.5, gs - 0.5],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "W":
        ax.plot([-0.5, -0.5], [exit_pos - 0.4, exit_pos + 0.4],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "E":
        ax.plot([gs - 0.5, gs - 0.5], [exit_pos - 0.4, exit_pos + 0.4],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")

    # Entity positions — use frame if provided, otherwise use initial from level_dict
    if frame is not None:
        pr, pc = frame["player"]
        m1r, m1c, m1alive = frame["mummy1"]
        m2r, m2c, m2alive = frame["mummy2"]
        sr, sc, salive = frame["scorpion"]
        gate_open = frame["gate_open"]
    else:
        pr, pc = level_dict["player"]
        m1r, m1c = level_dict["mummy1"]
        m1alive = True
        m2 = level_dict["mummy2"]
        m2r, m2c, m2alive = (m2[0], m2[1], True) if m2 else (0, 0, False)
        scorp = level_dict["scorpion"]
        sr, sc, salive = (scorp[0], scorp[1], True) if scorp else (0, 0, False)
        gate_open = False

    # Draw static entities first
    # Traps (X marks)
    for tr, tc in level_dict.get("traps", []):
        s = TRAP_SIZE
        ax.plot([tc - s, tc + s], [tr - s, tr + s], color="#8B0000", linewidth=2.5)
        ax.plot([tc - s, tc + s], [tr + s, tr - s], color="#8B0000", linewidth=2.5)

    # Key
    if level_dict.get("key"):
        kr, kc = level_dict["key"]
        ax.plot(kc, kr, marker="P", markersize=12, color="#FFD700",
                markeredgecolor="#B8860B", markeredgewidth=1.5, zorder=5)

    # Gate (vertical barrier on east edge of gate cell)
    if level_dict.get("gate"):
        gr, gc = level_dict["gate"]
        blocking = gate_open  # gate_open=True means blocking in Rust convention
        gate_color = "#8B0000" if blocking else "#90EE90"
        ax.plot([gc + 0.5, gc + 0.5], [gr - 0.35, gr + 0.35],
                color=gate_color, linewidth=4, solid_capstyle="round", zorder=6)

    # Draw entities
    # Mummy 1 (always present)
    mummy_color = "#D32F2F" if level_dict["flip"] else "#F5F5DC"
    mummy_edge = "#8B0000" if level_dict["flip"] else "#8B8B00"
    if m1alive:
        ax.add_patch(plt.Circle((m1c, m1r), ENTITY_RADIUS, color=mummy_color,
                                ec=mummy_edge, linewidth=2, zorder=8))
        ax.text(m1c, m1r, "M", ha="center", va="center", fontsize=9,
                fontweight="bold", color=mummy_edge, zorder=9)

    # Mummy 2
    if m2alive:
        ax.add_patch(plt.Circle((m2c, m2r), ENTITY_RADIUS, color=mummy_color,
                                ec=mummy_edge, linewidth=2, zorder=8))
        ax.text(m2c, m2r, "M", ha="center", va="center", fontsize=9,
                fontweight="bold", color=mummy_edge, zorder=9)

    # Scorpion
    if salive:
        ax.add_patch(plt.Circle((sc, sr), ENTITY_RADIUS, color="#FF8C00",
                                ec="#8B4500", linewidth=2, zorder=8))
        ax.text(sc, sr, "S", ha="center", va="center", fontsize=9,
                fontweight="bold", color="#8B4500", zorder=9)

    # Player (drawn last, on top)
    result = frame["result"] if frame else "ok"
    player_color = "#4FC3F7" if result == "ok" else ("#4CAF50" if result == "win" else "#EF5350")
    ax.add_patch(plt.Circle((pc, pr), ENTITY_RADIUS, color=player_color,
                            ec="#01579B", linewidth=2, zorder=10))
    ax.text(pc, pr, "P", ha="center", va="center", fontsize=9,
            fontweight="bold", color="#01579B", zorder=11)

    if title:
        ax.set_title(title, fontsize=11)


def animate_solution(level, interval=500, figsize=(5, 5)):
    """Animate the BFS solution of a level. Returns a JS-based animation for notebook display."""
    actions = mummymaze_rust.solve_actions(level)
    if actions is None:
        print("Level is unsolvable!")
        return None

    frames = mummymaze_rust.replay_actions(level, actions)
    level_dict = level.to_dict()
    action_names = ["N", "S", "E", "W", "Wait"]

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    def update(i):
        if i == 0:
            title = f"Step 0/{len(actions)} (initial)"
        else:
            title = f"Step {i}/{len(actions)} (action: {action_names[actions[i-1]]})"
        draw_maze(ax, level_dict, frames[i], title=title)

    anim = FuncAnimation(fig, update, frames=len(frames), interval=interval, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Test: draw a random original level
test_lev = all_levels[8][42]
fig, ax = plt.subplots(figsize=(5, 5))
draw_maze(ax, test_lev.to_dict(), title="Original level (gs=8, #42)")
plt.tight_layout()
plt.show()

In [ ]:
# Animate the BFS solution of that original level
animate_solution(test_lev, interval=600)

## 5. Compare: original vs GA-generated levels

In [ ]:
# Show 5 generated levels with their KDE target vs nearest original
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
ref_feats = original_features[8]
scales = feature_scales[8]

for i in range(min(5, len(generated_8))):
    lev, feat, fit, target = generated_8[i]

    # Find nearest original (to verify it's NOT a copy)
    normalized = (feat - ref_feats) / scales
    dists = np.linalg.norm(normalized, axis=1)
    nn_idx = dists.argmin()
    nn_dist = dists[nn_idx]

    draw_maze(axes[0, i], lev.to_dict(),
              title=f"Generated #{i+1}\nfit={fit:.3f}, nn_dist={nn_dist:.2f}")

    draw_maze(axes[1, i], all_levels[8][nn_idx].to_dict(),
              title=f"Nearest original\ndist={nn_dist:.3f}")

axes[0, 0].set_ylabel("Generated", fontsize=14, fontweight="bold")
axes[1, 0].set_ylabel("Nearest Original", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Animate a generated level's BFS solution
animate_solution(generated_8[0][0], interval=600)

## 6. Feature distribution comparison

Compare the feature distributions of the generated population vs originals.

In [ ]:
feature_names = ["BFS moves", "N states", "Log win prob", "Wall count"]
gen_feats = np.array([x[1] for x in generated_8])
target_feats = np.array([x[3] for x in generated_8])
orig_feats = original_features[8]

# 1D histograms
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, (ax, name) in enumerate(zip(axes, feature_names)):
    ax.hist(orig_feats[:, i], bins=30, alpha=0.5, label="Original", density=True, color="tab:blue")
    ax.hist(gen_feats[:, i], bins=15, alpha=0.6, label="Generated", density=True, color="tab:orange")
    ax.set_xlabel(name)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Feature Distributions: Original vs KDE-GA Generated (gs=8)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 2D scatter: generated levels overlaid on original KDE density
pairs = [(0, 1), (0, 2), (0, 3), (2, 3)]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, (i, j) in enumerate(pairs):
    ax = axes[idx]
    of = orig_feats

    # KDE density background
    data_2d = of[:, [i, j]].T
    kde_2d = gaussian_kde(data_2d)
    xi = np.linspace(of[:, i].min() - of[:, i].std() * 0.5,
                     of[:, i].max() + of[:, i].std() * 0.5, 60)
    yi = np.linspace(of[:, j].min() - of[:, j].std() * 0.5,
                     of[:, j].max() + of[:, j].std() * 0.5, 60)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = kde_2d(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
    ax.contourf(Xi, Yi, Zi, levels=15, cmap="Blues", alpha=0.6)

    # Original levels (faint)
    ax.scatter(of[:, i], of[:, j], s=2, alpha=0.1, color="gray", label="Original")
    # KDE targets (X markers)
    ax.scatter(target_feats[:, i], target_feats[:, j], s=40, marker="x",
               color="red", alpha=0.7, linewidths=1.5, label="KDE target")
    # Generated levels (dots)
    ax.scatter(gen_feats[:, i], gen_feats[:, j], s=30, color="tab:orange",
               edgecolors="black", linewidths=0.5, alpha=0.8, label="Generated")

    ax.set_xlabel(feature_names[i])
    ax.set_ylabel(feature_names[j])
    if idx == 0:
        ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Generated Levels in Feature Space (gs=8): KDE density + targets + results", fontsize=13)
plt.tight_layout()
plt.show()